In [ ]:
import os
import json

# 1. Authenticate
kaggle_creds = {"username":"YOUR_KAGGLE_USERNAME","key":"KGAT_ea1697ce904b91b747b4f8cf4684b6ea"}
with open('kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 2. Download
print("Downloading...")
!kaggle datasets download neehakurelli/google-speech-commands

# 3. List files to verify the exact zip name
print("Files in current directory:")
!ls *.zip

# 4. Force Unzip
# This finds any .zip file in the directory and unzips it into audio_data
!mkdir -p audio_data
!unzip -o *.zip -d ./audio_data > /dev/null

print("✅ Extraction attempt finished. Checking content...")
!ls ./audio_data | head -n 10

Downloading...
Dataset URL: https://www.kaggle.com/datasets/neehakurelli/google-speech-commands
License(s): unknown
100% 1.38G/1.38G [00:18<00:00, 80.3MB/s]

Files in current directory:
google-speech-commands.zip
✅ Extraction attempt finished. Checking content...
_background_noise_
bed
bird
cat
dog
down
eight
five
four
go


In [ ]:
import os
import glob
import torch
import librosa
import numpy as np
from torch.utils.data import Dataset, DataLoader

class AudioDataset(Dataset):
    def __init__(self, search_path, max_samples_per_class=1000):
        self.samples = []

        # Auto-discover where the actual data folders are
        search_pattern = os.path.join(search_path, "**/_background_noise_")
        found_paths = glob.glob(search_pattern, recursive=True)

        actual_root = os.path.dirname(found_paths[0]) if found_paths else search_path
        print(f"📁 Loading data from: {actual_root}")

        categories = [d for d in os.listdir(actual_root) if os.path.isdir(os.path.join(actual_root, d))]

        for cat in categories:
            cat_path = os.path.join(actual_root, cat)
            label = 0 if cat == '_background_noise_' else 1
            files = [f for f in os.listdir(cat_path) if f.endswith('.wav')]

            for f in files[:max_samples_per_class]:
                self.samples.append((os.path.join(cat_path, f), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            # Load 1-second audio
            y, sr = librosa.load(path, sr=16000, duration=1.0)
            if len(y) < 16000:
                y = np.pad(y, (0, 16000 - len(y)))

            # Convert to Mel Spectrogram
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
            mel_db = librosa.power_to_db(mel_spec, ref=np.max)
            mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)

            return torch.tensor(mel_db).unsqueeze(0), torch.tensor(label, dtype=torch.long)
        except:
            return torch.zeros((1, 64, 32)), torch.tensor(label, dtype=torch.long)

full_dataset = AudioDataset('./audio_data')

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_set, val_set = torch.utils.data.random_split(full_dataset, [train_size, val_size])

# num_workers=0 is CRITICAL to prevent Colab from freezing
train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=64, shuffle=False, num_workers=0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Ready: {len(train_set)} Training | {len(val_set)} Validation | Device: {device}")

📁 Loading data from: ./audio_data
🚀 Ready: 24004 Training | 6002 Validation | Device: cuda


In [ ]:
import torch.nn as nn

class AudioCNN(nn.Module):
    def __init__(self):
        super(AudioCNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.fc(self.conv(x))

model = AudioCNN().to(device)
print("✅ Model Architecture Loaded")

✅ Model Architecture Loaded


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 30
print("Starting Training...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation at the end of the epoch
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | Val Acc: {100*correct/total:.2f}%")

Starting Training...
Epoch 1/3 | Loss: 0.0152 | Val Acc: 99.98%
Epoch 2/3 | Loss: 0.0025 | Val Acc: 99.98%
Epoch 3/3 | Loss: 0.0020 | Val Acc: 99.98%


In [ ]:
from google.colab import files

# Save weights locally in Colab
model_filename = "audio_detection_model.pt"
torch.save(model.state_dict(), model_filename)
print(f"✅ Weights successfully saved as {model_filename}")

# Trigger download to your actual computer
files.download(model_filename)

✅ Weights successfully saved as audio_detection_model.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>